# Progressive Historical Glyph Dataset Generator
### Layer 3 Orchestration — 12-Stage Curriculum Pipeline

This notebook orchestrates the progressive generation of a high-quality historical glyph dataset for OCR / YOLO object detection training.

**Key Architecture Layers:**
- **Layer 1**: `historical_glyph_studio` (SVG parsing, material simulation, physical rendering, annotation)
- **Layer 2**: `historical_glyph_curriculum` (12 stages, 12 concepts/stage, progressive difficulty, validation)
- **Layer 3**: Google Colab Orchestration (interactive approval gates, checkpointing, token-safe Git sync)


In [ ]:
# @title 1. Environment & Hardware Diagnostics
import platform, sys, os, shutil
print(f'Python: {sys.version}')
print(f'Platform: {platform.platform()}')
print(f'Working dir: {os.getcwd()}')

try:
    import torch
    print(f'PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU Device: {torch.cuda.get_device_name(0)}')
except ImportError:
    print('PyTorch: Not installed')

try:
    import psutil
    mem = psutil.virtual_memory()
    print(f'System RAM: {mem.total / (1024**3):.1f} GB total, {mem.available / (1024**3):.1f} GB available')
except ImportError:
    pass

total, used, free = shutil.disk_usage(os.getcwd())
print(f'Disk Space: {free / (1024**3):.1f} GB free of {total / (1024**3):.1f} GB total')


In [ ]:
# @title 2. Install Engine & Dependencies
import subprocess, sys

GITHUB_REPO = 'https://github.com/Emran025/old-permic-ocr-lab'
BRANCH = 'colab-checkpoints'

print('Installing dependencies & packages from GitHub branch...')
cmd = [
    sys.executable, '-m', 'pip', 'install', '-q',
    f'git+{GITHUB_REPO}@{BRANCH}',
    'svglib', 'reportlab', 'psutil', 'tqdm', 'ipywidgets', 'scikit-image', 'opencv-python-headless'
]
subprocess.run(cmd, check=True)
print('✓ Installation complete.')


In [ ]:
# @title 3. Secure GitHub Authentication
# ═══════════════════════════════════════════════════════════════
# GitHub Token Setup (Required for pushing dataset checkpoints)
# Token is NEVER printed, logged, or written to disk.
# ═══════════════════════════════════════════════════════════════

GITHUB_TOKEN = None

# 1. Colab Secrets (Recommended)
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    if GITHUB_TOKEN:
        print('✓ GitHub Token loaded securely from Colab Secrets.')
except Exception:
    pass

# 2. Environment Variable
if not GITHUB_TOKEN:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')
    if GITHUB_TOKEN:
        print('✓ GitHub Token loaded from environment variable.')

# 3. Interactive Secure Prompt (Fallback)
if not GITHUB_TOKEN:
    import getpass
    print('Please enter your GitHub Personal Access Token (repo scope):')
    GITHUB_TOKEN = getpass.getpass('GitHub Token: ')
    if GITHUB_TOKEN:
        print('✓ GitHub Token entered securely.')

if not GITHUB_TOKEN:
    print('⚠ Warning: No GitHub Token provided. Git commits and pushes will be disabled in this session.')
else:
    print('✓ GitHub authentication configured.')


In [ ]:
# @title 4. Clone or Verify Repository
from pathlib import Path
import os, subprocess

GITHUB_REPO = 'https://github.com/Emran025/old-permic-ocr-lab'
BRANCH = 'colab-checkpoints'

# Check if running directly inside the cloned repo or in Colab /content
if Path('historical_glyph_studio').exists() and Path('font/svg').exists():
    REPO_DIR = Path(os.getcwd()).resolve()
    print(f'Running inside existing repository: {REPO_DIR}')
else:
    REPO_DIR = Path('/content/old-permic-ocr-lab').resolve()
    if REPO_DIR.exists():
        print(f'Repository directory exists at {REPO_DIR}. Pulling latest changes...')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
    else:
        print(f'Cloning branch {BRANCH} to {REPO_DIR}...')
        from historical_glyph_curriculum.github_sync import GitManager
        GitManager.clone(url=GITHUB_REPO, branch=BRANCH, target_dir=REPO_DIR, token=GITHUB_TOKEN)
        print(f'✓ Cloned successfully to {REPO_DIR}')

print(f'Working repository root: {REPO_DIR}')


In [ ]:
# @title 5. Verify Glyph Sources & Package Structure
GLYPH_ROOT = REPO_DIR / 'font' / 'svg'

required_dirs = [
    REPO_DIR / 'historical_glyph_studio',
    REPO_DIR / 'historical_glyph_curriculum',
    GLYPH_ROOT
]

all_ok = True
for p in required_dirs:
    status = '✓ FOUND' if p.exists() else '✗ MISSING'
    print(f'  {status}: {p.name}')
    if not p.exists():
        all_ok = False

svg_files = list(GLYPH_ROOT.rglob('*.svg')) if GLYPH_ROOT.exists() else []
print(f'\nTotal SVG glyph files discovered: {len(svg_files)}')

if not all_ok or len(svg_files) == 0:
    raise RuntimeError('Repository verification failed. Ensure font/svg directory is populated.')
print('✓ Project structure verified successfully.')


In [ ]:
# @title 6. Initialize & Smoke-Test Historical Glyph Studio
import sys
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from historical_glyph_studio import GlyphStudio

studio = GlyphStudio(glyph_root=GLYPH_ROOT, canonical_size=(128, 128))
print(studio.repository_summary())

available_chars = studio.available_chars()
available_families = studio.available_families()
print(f'Discovered {len(available_chars)} unique characters across {len(available_families)} families.')

# Quick smoke test render
if available_chars:
    test_sample = studio.render(available_chars[0], operation='engraved', seed=42)
    print(f'✓ Smoke test successful: Rendered {test_sample.image.shape} RGB image with {len(test_sample.annotation.boxes)} annotation box(es).')


In [ ]:
# @title 7. Curriculum & Generation Configuration
# ═══════════════════════════════════════════════════════════════
# Generation Configuration Controls
# ═══════════════════════════════════════════════════════════════

GENERATION_MODE = 'dev'  # Options: 'dev' (quick test), 'medium' (500/stage), 'full' (production)

SAMPLES_PER_STAGE = {
    'dev': 24,       # Quick verification across all 12 concepts
    'medium': 500,   # Moderate dataset
    'full': None     # Uses default_samples defined in each stage specification
}

STAGES_TO_RUN = list(range(1, 13))  # [1, 2, ..., 12]
GLOBAL_SEED = 2025
AUTO_APPROVE = (GENERATION_MODE == 'dev')

DATASET_ROOT = REPO_DIR / 'datasets'
PREVIEWS_ROOT = REPO_DIR / 'previews'
METADATA_ROOT = REPO_DIR / 'metadata'

for d in [DATASET_ROOT, PREVIEWS_ROOT, METADATA_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Generation Mode:    {GENERATION_MODE}')
print(f'Samples per stage:  {SAMPLES_PER_STAGE[GENERATION_MODE] or "Stage Defaults (1500-3000)"}')
print(f'Stages to execute:  {STAGES_TO_RUN}')
print(f'Auto Approval:      {AUTO_APPROVE}')
print(f'Target dataset dir: {DATASET_ROOT}')


In [ ]:
# @title 8. Hardware Resource Detection & Worker Tuning
from historical_glyph_curriculum.resources import detect_resources, print_resource_report, auto_tune

profile = detect_resources()
print_resource_report(profile)

WORKERS, BATCH_SIZE = auto_tune(
    profile,
    override_workers=None,   # Set integer to override
    override_batch=None      # Set integer to override
)
print(f'→ Execution Plan: {WORKERS} parallel worker process(es), batch size = {BATCH_SIZE}')


In [ ]:
# @title 9. Setup Curriculum Engine & Pipeline Components
from historical_glyph_curriculum import STAGES, get_stage, GenerationPlan
from historical_glyph_curriculum.parallel import CurriculumExecutor
from historical_glyph_curriculum.preview import display_grid_in_colab, select_preview_samples, build_preview_grid
from historical_glyph_curriculum.validation import DatasetValidator
from historical_glyph_curriculum.metadata import (
    StageManifest, save_stage_manifest, load_stage_manifest, build_master_manifest, generate_curriculum_report, print_stage_summary
)
from historical_glyph_curriculum.github_sync import GitManager

executor = CurriculumExecutor(
    glyph_root=GLYPH_ROOT,
    workers=WORKERS,
    batch_size=BATCH_SIZE,
    canonical_size=(128, 128)
)

validator = DatasetValidator()
git = GitManager(
    repo_dir=REPO_DIR,
    remote_url='https://github.com/Emran025/old-permic-ocr-lab',
    branch='colab-checkpoints'
)
git.configure_identity(name='Old Permic OCR Lab Bot', email='bot@ocr-lab.ai')

completed_manifests = []
print('✓ Pipeline components successfully initialized.')


In [ ]:
# @title 10. Interactive Approval Gate Helper
def approval_gate(prompt_text: str, auto: bool = AUTO_APPROVE) -> bool:
    '''Interactive prompt for researcher approval before proceeding or committing.'''
    if auto:
        print(f'  [AUTO-APPROVED]: {prompt_text}')
        return True
    try:
        import ipywidgets as widgets
        from IPython.display import display
        out = widgets.Output()
        res = [None]
        btn_yes = widgets.Button(description='✓ Approve', button_style='success', icon='check')
        btn_no = widgets.Button(description='✗ Reject', button_style='danger', icon='times')
        def on_y(b):
            res[0] = True
            with out: print('✓ Approved by user.')
        def on_n(b):
            res[0] = False
            with out: print('✗ Rejected by user.')
        btn_yes.on_click(on_y)
        btn_no.on_click(on_n)
        display(widgets.HTML(f'<b>{prompt_text}</b>'))
        display(widgets.HBox([btn_yes, btn_no]))
        display(out)
        import time
        for _ in range(300):
            if res[0] is not None:
                return res[0]
            time.sleep(1)
        print('Approval timeout. Defaulting to Reject.')
        return False
    except Exception:
        ans = input(f'{prompt_text} [y/n]: ').strip().lower()
        return ans in ('y', 'yes', '1', 'true')


In [ ]:
# @title 11. Stage Execution Function
import time, json

def run_stage(stage_id: int, token: str | None) -> StageManifest | None:
    stage_def = get_stage(stage_id)
    n_samples = SAMPLES_PER_STAGE[GENERATION_MODE] or stage_def.default_samples

    print(f'\n{"═"*65}')
    print(f'  STAGE {stage_id:02d}: {stage_def.name}')
    print(f'  Description: {stage_def.description}')
    print(f'  Target Samples: {n_samples:,} | Canvas: {stage_def.canvas_size}')
    print(f'  Difficulty Score: {stage_def.difficulty.overall_score():.2f} / 1.00')
    print(f'{"═"*65}')

    stage_dir = DATASET_ROOT / stage_def.stage_dir_name
    preview_dir = PREVIEWS_ROOT / stage_def.stage_dir_name
    preview_dir.mkdir(parents=True, exist_ok=True)
    state_path = stage_dir / 'state.json'
    manifest_path = stage_dir / 'manifest.json'

    # Check if already completed and committed
    if state_path.exists():
        try:
            st = json.loads(state_path.read_text(encoding='utf-8'))
            if st.get('committed') and manifest_path.exists():
                print(f'✓ Stage {stage_id:02d} is already generated and committed. Loading manifest...')
                return load_stage_manifest(manifest_path)
        except Exception:
            pass

    # 1. Preview Phase
    preview_count = min(24, n_samples)
    print(f'\n[1/4] Generating {preview_count} preview samples across 12 concepts...')
    preview_plan = GenerationPlan.build(
        stage_def, available_chars, preview_count, stage_dir, global_seed=GLOBAL_SEED
    )
    t0 = time.time()
    executor.generate_stage(preview_plan, state_path=stage_dir / 'preview_state.json')
    print(f'Preview samples generated in {time.time() - t0:.1f}s.')

    preview_samples = select_preview_samples(stage_dir / 'images', n=preview_count)
    if preview_samples:
        display_grid_in_colab(preview_samples, n_cols=6, title=f'Stage {stage_id:02d}: {stage_def.name} (Preview)')

    if not approval_gate(f'Approve Stage {stage_id:02d} visual generation strategy to proceed with full dataset?'):
        print(f'Stage {stage_id:02d} rejected by user. Halting.')
        return None

    # 2. Full Generation Phase
    print(f'\n[2/4] Generating full dataset ({n_samples:,} samples)...')
    if (stage_dir / 'preview_state.json').exists():
        (stage_dir / 'preview_state.json').unlink()

    full_plan = GenerationPlan.build(
        stage_def, available_chars, n_samples, stage_dir, global_seed=GLOBAL_SEED
    )
    t_start = time.time()
    summary = executor.generate_stage(
        full_plan,
        state_path=state_path,
        progress_callback=lambda d, t, c: print(f'  Concept [{c}]: {d}/{t} completed', end='\r') if d % 10 == 0 or d == t else None
    )
    elapsed = time.time() - t_start
    print(f'\nFull generation finished in {elapsed:.1f}s ({summary["total_images"] / max(elapsed, 0.001):.1f} img/s).')

    # 3. Quality Validation Phase
    print('\n[3/4] Validating dataset annotations and images...')
    report = validator.validate(stage_dir / 'images', stage_dir / 'labels')
    print(report.summary())

    final_previews = select_preview_samples(stage_dir / 'images', n=24, strategy='spread')
    if final_previews:
        grid_img = build_preview_grid(final_previews, n_cols=6, title=f'Stage {stage_id:02d} Final Samples')
        grid_img.save(preview_dir / 'preview_grid.png')
        display_grid_in_colab(final_previews, n_cols=6, title=f'Stage {stage_id:02d} Final Samples')

    # 4. Approval & GitHub Commit Phase
    print(f'\n[4/4] Commit & Push Checkpoint for Stage {stage_id:02d}...')
    commit_hash = None
    if token and approval_gate(f'Commit & Push Stage {stage_id:02d} ({summary["total_images"]} images) to GitHub branch "colab-checkpoints"?'):
        try:
            commit_hash = git.commit_and_push_stage(
                stage_id=stage_id,
                stage_name=stage_def.name,
                dataset_dir=stage_dir,
                token=token,
                previews_dir=preview_dir,
                metadata_dir=METADATA_ROOT
            )
        except Exception as e:
            print(f'Push error: {e}')

    manifest = StageManifest(
        stage_id=stage_id,
        stage_name=stage_def.name,
        total_images=summary['total_images'],
        class_distribution=summary['class_distribution'],
        materials_used=summary['materials_used'],
        families_used=summary['families_used'],
        resolution_range=(0, 0),
        seed=GLOBAL_SEED,
        approved=True,
        commit_hash=commit_hash,
        generation_time_seconds=elapsed,
    )
    save_stage_manifest(manifest, stage_dir)

    st = json.loads(state_path.read_text()) if state_path.exists() else {}
    st['committed'] = bool(commit_hash)
    state_path.write_text(json.dumps(st, indent=2))

    print_stage_summary(manifest)
    return manifest


In [ ]:
# @title Stage 01 — Clean Isolated Glyphs
if 1 in STAGES_TO_RUN:
    m_01 = run_stage(1, GITHUB_TOKEN)
    if m_01:
        completed_manifests.append(m_01)
else:
    print('Stage 01 skipped (not in STAGES_TO_RUN).')


In [ ]:
# @title Stage 02 — Material Variation
if 2 in STAGES_TO_RUN:
    m_02 = run_stage(2, GITHUB_TOKEN)
    if m_02:
        completed_manifests.append(m_02)
else:
    print('Stage 02 skipped (not in STAGES_TO_RUN).')


In [ ]:
# @title Stage 03 — Controlled Degradation
if 3 in STAGES_TO_RUN:
    m_03 = run_stage(3, GITHUB_TOKEN)
    if m_03:
        completed_manifests.append(m_03)
else:
    print('Stage 03 skipped (not in STAGES_TO_RUN).')


In [ ]:
# @title Stage 04 — Discriminative-Aware Occlusion
if 4 in STAGES_TO_RUN:
    m_04 = run_stage(4, GITHUB_TOKEN)
    if m_04:
        completed_manifests.append(m_04)
else:
    print('Stage 04 skipped (not in STAGES_TO_RUN).')


In [ ]:
# @title Stage 05 — Geometric & Camera Variation
if 5 in STAGES_TO_RUN:
    m_05 = run_stage(5, GITHUB_TOKEN)
    if m_05:
        completed_manifests.append(m_05)
else:
    print('Stage 05 skipped (not in STAGES_TO_RUN).')


In [ ]:
# @title Stage 06 — Multiple Glyphs
if 6 in STAGES_TO_RUN:
    m_06 = run_stage(6, GITHUB_TOKEN)
    if m_06:
        completed_manifests.append(m_06)
else:
    print('Stage 06 skipped (not in STAGES_TO_RUN).')


In [ ]:
# @title Stage 07 — Glyph Groups & Sequences
if 7 in STAGES_TO_RUN:
    m_07 = run_stage(7, GITHUB_TOKEN)
    if m_07:
        completed_manifests.append(m_07)
else:
    print('Stage 07 skipped (not in STAGES_TO_RUN).')


In [ ]:
# @title Stage 08 — Lines & Text-Like Structures
if 8 in STAGES_TO_RUN:
    m_08 = run_stage(8, GITHUB_TOKEN)
    if m_08:
        completed_manifests.append(m_08)
else:
    print('Stage 08 skipped (not in STAGES_TO_RUN).')


In [ ]:
# @title Stage 09 — Multi-Line Text
if 9 in STAGES_TO_RUN:
    m_09 = run_stage(9, GITHUB_TOKEN)
    if m_09:
        completed_manifests.append(m_09)
else:
    print('Stage 09 skipped (not in STAGES_TO_RUN).')


In [ ]:
# @title Stage 10 — Historical Document Structure
if 10 in STAGES_TO_RUN:
    m_10 = run_stage(10, GITHUB_TOKEN)
    if m_10:
        completed_manifests.append(m_10)
else:
    print('Stage 10 skipped (not in STAGES_TO_RUN).')


In [ ]:
# @title Stage 11 — Severe Historical Degradation
if 11 in STAGES_TO_RUN:
    m_11 = run_stage(11, GITHUB_TOKEN)
    if m_11:
        completed_manifests.append(m_11)
else:
    print('Stage 11 skipped (not in STAGES_TO_RUN).')


In [ ]:
# @title Stage 12 — Realistic Mixed Historical Scenes
if 12 in STAGES_TO_RUN:
    m_12 = run_stage(12, GITHUB_TOKEN)
    if m_12:
        completed_manifests.append(m_12)
else:
    print('Stage 12 skipped (not in STAGES_TO_RUN).')


In [ ]:
# @title 12. Master Manifest & Comprehensive Curriculum Report
if completed_manifests:
    print(f'Compiling Master Manifest across {len(completed_manifests)} stage(s)...')
    master = build_master_manifest(
        completed_manifests,
        output_path=METADATA_ROOT / 'master_manifest.json'
    )
    report_text = generate_curriculum_report(
        master,
        output_path=METADATA_ROOT / 'curriculum_report.md'
    )
    print(report_text)

    if GITHUB_TOKEN and approval_gate('Commit Master Manifest & Final Report to GitHub?'):
        git.stage_files([METADATA_ROOT, PREVIEWS_ROOT])
        hash_val = git.commit('curriculum: compile master manifest and comprehensive generation report')
        git.push_with_auth(GITHUB_TOKEN)
        print(f'✓ Master curriculum checkpoint committed and pushed: {hash_val}')
else:
    print('No stages completed in this session.')
